In [1]:
from hrs_botany.spectral_cleaning import SpectralCleanConfig, SpectralCleaner

ModuleNotFoundError: No module named 'hrs_botany'

In [ ]:
# 1) Load wavelengths from any one EnMAP JSON sidecar (make sure order matches bands)
with open("path/to/one_scene/…SPECTRAL_IMAGE_COG.json") as f:
    meta = json.load(f)
# adjust the path below to wherever wavelengths live in your JSON:
wl_nm = np.array(meta["sensor"]["spectral"]["wavelength_center_nm"], dtype=float)

# 2) Configure
cfg = SpectralCleanConfig(
    wl_nm=wl_nm,
    keep_ranges_nm=((430,1330),(1460,1780),(1960,2450)),
    hampel_window=5, hampel_nsig=4.0,
    sg_window=7, sg_poly=2,
    scene_robust_scale=True,
    groupby_col="scene_id",
    min_coverage_keep=0.7,                 # tighten as you like
    qa_drop_cols=["cloud", "shadow", "snow", "saturated"],  # if present
    stats_path=Path("artifacts/spectral_clean_stats.json"),
)

cleaner = SpectralCleaner(cfg)

# 3) Training/assembly split:
#    - Fit + transform on your training build
df_train_clean = cleaner.fit_transform(df_train)

#    - Save stats (already saved if cfg.stats_path is set)
cleaner.save_stats()

# 4) Apply to eval/test later (reuses per-scene stats; unseen scenes get fallback stats)
cleaner2 = SpectralCleaner(cfg)
cleaner2.load_stats()
df_test_clean = cleaner2.transform(df_test)
